# Capstone-opdracht · Alles samenbrengen
### Regressie-training · Antwoordmodel-versie

Dit is de afsluitende opdracht. Je past nu zelf de hele workflow toe die je vandaag hebt geleerd:
van probleem en data tot modelkeuze, evaluatie en conclusie.

**De stappen:**
1. Kies een dataset en formuleer een duidelijke voorspelvraag
2. Verken en bereid de data voor
3. Maak een eerlijke train/test-opzet
4. Bouw een **baseline** en minstens **twee** modellen
5. Evalueer eerlijk en toets de aannames
6. Kies een model en onderbouw je keuze (korte presentatie)

> **Werkvorm:** in groepjes · rest van de middag · sluit af met een presentatie van 5 minuten.


> **Voor de trainer:** dit antwoordmodel werkt de capstone volledig uit op de Diabetes-dataset,
> als referentie en als voorbeeld om met de groep te bespreken. Cursisten mogen een eigen dataset kiezen;
> de stappen en de redenering blijven hetzelfde.

## 0. Voorbereiding
Imports. Hier zit alles in wat je voor de hele workflow nodig hebt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (7, 4)
print("Bibliotheken geladen.")

## 1. Dataset & voorspelvraag

We gebruiken hier de Diabetes-dataset. **Voorspelvraag:** kunnen we de ziekteprogressie na één jaar
voorspellen op basis van de medische metingen?

In [ ]:
# ANTWOORD - voorbeelddataset
from sklearn.datasets import load_diabetes
data = load_diabetes(as_frame=True)
df = data.frame
target_kolom = "target"
print("Dataset geladen:", df.shape)
df.head()

## 2. Data verkennen
Bekijk de data voordat je modelleert: vorm, samenvatting en de verdeling van de target.

In [ ]:
# ANTWOORD
print(df.describe())
df[target_kolom].hist(bins=40, color="#4473C5")
plt.title("Verdeling van de target")
plt.show()

## 3. Target/features scheiden en splitsen

In [ ]:
# ANTWOORD
y = df[target_kolom]
X = df.drop(columns=[target_kolom])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

## 4. Baseline + minstens twee modellen
Begin altijd met een **baseline** (het gemiddelde voorspellen). Voeg daarna minstens twee echte modellen toe.
Hieronder een handige opzet: een dictionary van modellen die we straks in één keer evalueren.

In [ ]:
# ANTWOORD
modellen = {
    "Baseline (gemiddelde)": DummyRegressor(strategy="mean"),
    "Lineaire regressie": LinearRegression(),
    "Random forest": RandomForestRegressor(n_estimators=100, random_state=42),
}

## 5. Evalueren
We trainen elk model en berekenen MAE, RMSE en R² op de test-set. De evaluatie-lus is al voor je geschreven.

In [ ]:
# Evaluatie-lus (al ingevuld): traint elk model en verzamelt de scores
resultaten = []
for naam, model in modellen.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    resultaten.append({
        "model": naam,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })

resultaten = pd.DataFrame(resultaten).set_index("model")
print(resultaten.round(3))

resultaten["R2"].plot(kind="bar", color="#4473C5")
plt.ylabel("R² op de test-set"); plt.title("Modelvergelijking")
plt.xticks(rotation=20, ha="right"); plt.axhline(0, color="black", linewidth=0.8)
plt.show()

### TODO 5 — Toets de aannames van je beste model
Afhankelijk van je beste model, doe minstens één diagnostische check. Bijvoorbeeld:
- Lineair model → maak een **residuenplot** (zie opdracht 2)
- Random forest → bekijk de **feature importance** (zie opdracht 5)

> Kies de check die bij je model past en voer hem hieronder uit.

In [ ]:
# ANTWOORD - residuenplot voor het lineaire model als voorbeeld
lin = LinearRegression().fit(X_train, y_train)
residuen = y_test - lin.predict(X_test)
plt.scatter(lin.predict(X_test), residuen, color="#4473C5", alpha=0.7, edgecolor="white")
plt.axhline(0, color="#D97733", linewidth=2)
plt.xlabel("Voorspelde waarde"); plt.ylabel("Residu")
plt.title("Residuenplot - lineair model")
plt.show()

## 6. Modelkeuze & conclusie

In [ ]:
# ANTWOORD
beste = resultaten["R2"].idxmax()
print(f"Beste model op R²: {beste}  (R² = {resultaten.loc[beste, 'R2']:.3f})")

**Conclusie (voorbeeld):** het lineaire model en het random forest presteren vergelijkbaar en verslaan
de baseline duidelijk. Het lineaire model wint hier nipt op R² én is beter interpreteerbaar, dus dat zou
hier de voorkeur hebben. Bij een sterker niet-lineair probleem zou het random forest waarschijnlijk verder
uitlopen.

## 7. Presenteer
Bereid een korte presentatie (5 min) voor met:

- Je voorspelvraag en dataset
- Welke modellen je hebt vergeleken en hoe ze scoorden
- Je modelkeuze en de onderbouwing
- Eén ding dat je verraste of dat je de volgende keer anders zou doen

> Beoordeel elkaar vooral op de **aanpak** en de redenering, niet alleen op de hoogste score.

---
### Toelichting voor de trainer
- Met de Diabetes-dataset: **baseline R² ≈ 0**, **lineair ≈ 0,45**, **random forest ≈ 0,44**.
- Mooi gesprekspunt: het simpelere, beter interpreteerbare lineaire model wint hier nipt — *complexer is
  niet altijd beter*. Dit is precies de boodschap van blok 6.
- Let bij groepjes met een eigen dataset op de klassieke valkuilen: data-lekkage, verkeerde validatie
  (shuffle bij tijdreeksen), en blind staren op één metriek.
- De evaluatie-lus is bewust herbruikbaar gemaakt, zodat cursisten makkelijk extra modellen kunnen toevoegen.